<a href="https://colab.research.google.com/github/Tomoki4e7e/DLBasic2026_Final_Assignment_VQA/blob/develop/DL_Basic_2026_Spring_Competition_VQA_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning 基礎講座　最終課題: VQA

## 概要
画像と質問から，回答を予測するタスクです．
- サンプル数: 訓練 19,873 サンプル，テスト 4,969 サンプル
- 入力: 画像データ（RGB，サイズは画像によって異なります），質問文（系列長はサンプルごとに異なります）
- 出力: 回答文（系列長はサンプルごとに異なります）
- 評価指標: VQA での評価指標（[こちら
](https://visualqa.org/evaluation.html)を参照）を利用しています．

### データセット ([VizWiz 2023 edition](https://www.kaggle.com/datasets/nqa112/vizwiz-2023-edition)) の詳細
- 24,842 枚の画像データセットと，各画像に対する 1 つの質問文と 10 人の回答者による回答文から構成されます．
  - 10 人の回答は全て同じとは限りません．
- 24.842 サンプルのうち，80 % (19.873) が訓練データ (train)，20 % (4969) がテストデータ (val) として与えられます．
  - テストデータに対する回答文を正解ラベルとし，配布していません．
  - データ提供元とは異なるデータ分割になっています．

### タスクの詳細
- 本コンペティションでは，与えられた画像と質問文に対して，適切な回答文を出力するモデルを作成していただきます．
- 評価は [VQA](https://visualqa.org/index.html) (Visual Question Answering) に基づいて，以下の式で計算されます．

$$\text{Acc}(ans) = \text{min}(\frac{humans \; that \; said \; ans}{3}, 1)$$

- 1 つのデータに対し， 10 人の回答のうち 9 人の回答を選択し上記の式で性能評価した， 10 パターンの Acc の平均をそのデータに対する Acc とします．
- 予測結果と正解ラベルを比較する前に，回答を lowercase にする，冠詞は削除するなどの前処理を行っています（[詳細](https://visualqa.org/evaluation.html)）．

## 考えられる工夫の例
- 事前学習モデルの fine-tuning
    - 画像特徴量，言語特徴量を取得するときに，事前学習モデルを fine-tuning することで性能向上が見込めます（今回のタスクと大きく異なるデータセットでの事前学習では効果が小さい可能性がありますので注意しましょう）．
- 質問文の表現
    - ベースラインでは，質問文をモデルに入力する際に，one-hot ベクトルにしています．これを tokenizer 等を利用して分散表現にすることで，モデル学習しやすくなります．
- ソフトラベルの利用
    - ベースラインでは 10 人の回答の中で最も多かった回答を正解ラベルとして訓練しています．この点を各回答の頻度に合わせてソフトラベルを利用することで，より多くの情報を利用して学習が可能になります．
- 画像の前処理
    - 画像の前処理には形状を同じにする Resize のみを利用しています．「畳み込みニューラルネットワーク」，「深層学習と画像認識」等で紹介されていたデータ拡張を追加することで，汎化性能の向上が見込めます．

## 修了要件を満たす条件
- ベースラインでは，omnicampus 上での性能評価において， 49.9% となります．したがって，ベースラインを超える 49.9% を超えた提出のみ，修了要件として認めます．
- ベースラインから改善を加えることで， 60% に性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

### データの準備
データをダウンロードした際に，google drive したため，利用するために google drive をマウントする必要があります．また， drive 上で展開することができないため，/content ディレクトリ下にコピーし "data.zip" を展開します．  
google drive 上に "data.zip" が配置されていない場合は実行できません．google drive 上に "data.zip" (**12GB**) を配置することが可能であれば，"data_download.ipynb" を先に実行してください．難しい場合は，omnicampus 演習環境を利用してください．．



In [ ]:
# omnicampus 上では 4 セル目まで実行不要
# ドライブのマウント
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base_dir = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment"
zip_path = os.path.join(base_dir, "data.zip")
local_zip_path = "/content/data.zip"
local_data_dir = "/content/data"

# 既存のシンボリックリンクがあれば削除
if os.path.islink(local_data_dir):
    os.unlink(local_data_dir)
    print("古いシンボリックリンクを削除しました。")

if os.path.exists(local_data_dir):
    print("既に解凍済みのローカルデータが存在します。スキップします。")
else:
    print("Google Driveからデータをローカル環境にコピーします...")
    !cp "{zip_path}" "{local_zip_path}"
    print("コピー完了。解凍を開始します...")
    !unzip -q "{local_zip_path}" -d "/content/"
    print("ローカル環境へのデータ展開が完了しました。")


Google Driveからデータをローカル環境にコピーします...
コピー完了。解凍を開始します...
ローカル環境へのデータ展開が完了しました。


In [ ]:
# カレントディレクトリ下のファイル群を確認
# data が水色（シンボリックリンク）で表示されていれば問題ありません
%ls -l


total 11682740
drwxr-xr-x 4 root root        4096 Jun  7 07:25 data/
-rw------- 1 root root 11963107886 Jun 23 15:18 data.zip
drwx------ 5 root root        4096 Jun 23 15:13 drive/
drwxr-xr-x 1 root root        4096 Jun  4 13:39 sample_data/


In [ ]:
# データダウンロード用の notebook にてgoogle drive への保存後，
# 反映に時間がかかる可能性がありますので，google drive のマウント後，
# data.zip がディレクトリ内にあることを確認してから実行してください．
# data.zip を /content 下にコピーする
# !cp "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/data.zip" "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment"

In [ ]:
# カレントディレクトリ下のファイル群を確認
# data.zip が表示されれば問題ないです
# %ls

In [ ]:
# データを解凍する
# !unzip data.zip

### 1. import library

In [ ]:
!pip install transformers
!pip install sentencepiece

In [ ]:
import re
import random
import time
from statistics import mode

from PIL import Image
import numpy as np
import pandas
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from collections import Counter
from transformers import AutoTokenizer, AutoModel

torch.cuda.is_available()

True

### 2. utils

In [ ]:
def set_seed(seed):
    """
    シードを固定する．

    Parameters
    ----------
    seed : int
        乱数生成に用いるシード値．
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def process_text(text):
    """
    入力文と回答のフォーマットを統一するための関数．

    Parameters
    ----------
    text : str
        入力文，もしくは回答．
    """
    # lowercase
    text = text.lower()

    # 数詞を数字に変換
    num_word_to_digit = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'ten': '10'
    }
    for word, digit in num_word_to_digit.items():
        text = text.replace(word, digit)

    # 小数点のピリオドを削除
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)

    # 冠詞の削除
    text = re.sub(r'\b(a|an|the)\b', '', text)

    # 短縮形のカンマの追加
    contractions = {
        "dont": "don't", "isnt": "isn't", "arent": "aren't", "wont": "won't",
        "cant": "can't", "wouldnt": "wouldn't", "couldnt": "couldn't"
    }
    for contraction, correct in contractions.items():
        text = text.replace(contraction, correct)

    # 句読点をスペースに変換
    text = re.sub(r"[^\w\s':]", ' ', text)

    # 句読点をスペースに変換
    text = re.sub(r'\s+,', ',', text)

    # 連続するスペースを1つに変換
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
class VQADataset(torch.utils.data.Dataset):
    """
    VQA データセットを扱うためのクラス．
    """
    def __init__(self, df_path, image_dir, transform=None, answer=True):
        self.transform = transform  # 画像の前処理
        self.image_dir = image_dir  # 画像ファイルのディレクトリ
        self.df = pandas.read_json(df_path)  # 画像ファイルのパス，question, answerを持つDataFrame
        self.answer = answer

        # --- 変更点: AutoTokenizerの利用 ---
        self.text_model_name = 'microsoft/deberta-v3-base' # 'microsoft/deberta-v3-base' などに変更可能
        self.tokenizer = AutoTokenizer.from_pretrained(self.text_model_name)

        self.answer2idx = {}
        self.idx2answer = {}

        if self.answer:
            # 回答に含まれる文章を辞書に追加
            for answers in self.df["answers"]:
                for answer in answers:
                    word = answer["answer"]
                    word = process_text(word)
                    if word not in self.answer2idx:
                        self.answer2idx[word] = len(self.answer2idx)
            self.idx2answer = {v: k for k, v in self.answer2idx.items()}  # 逆変換用の辞書(answer)

    def update_dict(self, dataset):
        """
        検証用データ，テストデータの辞書を訓練データの辞書に更新する．
        """
        self.answer2idx = dataset.answer2idx
        self.idx2answer = dataset.idx2answer

    def __getitem__(self, idx):
        """
        対応するidxのデータ（画像，質問，回答）を取得．
        """
        image = Image.open(f"{self.image_dir}/{self.df['image'][idx]}")
        image = self.transform(image)

        question_text = self.df["question"][idx]

        encoded = self.tokenizer(
            question_text,
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors='pt'
        )

        inputs_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        if self.answer:
            answers = [self.answer2idx[process_text(answer["answer"])] for answer in self.df["answers"][idx]]

            soft_label = torch.zeros(len(self.answer2idx))
            answer_counts = Counter(answers)

            for ans_id, count in answer_counts.items():
                soft_label[ans_id] = min(count / 3.0, 1.0)

            return image, inputs_ids, attention_mask, torch.Tensor(answers), soft_label
        else:
            return image, inputs_ids, attention_mask

    def __len__(self):
        return len(self.df)

In [ ]:
def VQA_criterion(batch_pred, batch_answers):
    """
    VQA タスクに用いられる評価関数．
    """
    total_acc = 0.

    for pred, answers in zip(batch_pred, batch_answers):
        acc = 0.
        for i in range(len(answers)):
            num_match = 0
            for j in range(len(answers)):
                if i == j:
                    continue
                if pred == answers[j]:
                    num_match += 1
            acc += min(num_match / 3, 1)
        total_acc += acc / 10

    return total_acc / len(batch_pred)

### 3. model

In [ ]:
class VQAModel(nn.Module):
    """
    VQA タスクを解くためのモデル例．
    """
    def __init__(self, n_answer: int):
        """
        コンストラクタ．

        Parameters
        ----------
        n_answer: int
            出力のクラス数
        """
        super().__init__()
        # 1. 事前学習済みVision Transformer (ViT-B/16) のロード
        self.vit = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
        # 分類ヘッドを無効化して特徴量抽出器として利用 (出力次元: 768)
        self.vit.heads = nn.Identity()

        # テキストエンコーダを元の microsoft/deberta-v3-base に戻す
        text_model_name = 'microsoft/deberta-v3-base'
        self.text_encoder = AutoModel.from_pretrained(text_model_name)

        # Cross-Attention: embed_dim をテキストの hidden_size に合わせる
        self.cross_attn = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        # Attention を通した特徴量と画像特徴量（射影後）を結合して出力層へ
        self.fc = nn.Sequential(
            nn.Linear(768 * 2, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(512, n_answer)
        )

    def forward(self, image, input_ids, attention_mask):
        # 画像の特徴量を取得: [batch_size, image_feat_dim]
        image_feature = self.vit(image).to(torch.float32)

        # テキストの系列全体の特徴量を取得: [batch_size, seq_len, text_hidden]
        self.text_encoder = self.text_encoder.float()
        with torch.amp.autocast('cuda', enabled=False):
            text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            text_seq = text_outputs.last_hidden_state.to(torch.float32)

        # --- Cross-Attentionの適用 ---
        # 画像特徴量をQueryとして [batch_size, 1, text_hidden] の形状にする
        query = image_feature.unsqueeze(1)
        # テキスト特徴量をKey, Valueとする
        key = text_seq
        value = text_seq

        # BERTのpadding部分をAttentionから除外するマスクを作成
        key_padding_mask = (attention_mask == 0)
        key_padding_mask[:, 0] = False # 先頭の[CLS]トークンだけは絶対にマスクしない

        # Attentionの計算: attn_outputは [batch_size, 1, text_hidden]
        attn_output, _ = self.cross_attn(query, key, value, key_padding_mask=key_padding_mask)
        attn_output = attn_output.squeeze(1) # [batch_size, text_hidden] に戻す

        # 元の画像特徴量（射影後）とAttentionの出力を結合して予測
        x = torch.cat([image_feature, attn_output], dim=1)
        x = self.fc(x)

        return x

### 4. train

In [ ]:
def train(model, dataloader, optimizer, criterion, device):
    """
    学習用の関数．
    """
    model.train()

    total_loss = 0
    total_acc = 0
    simple_acc = 0

    start = time.time()
    for image, input_id, attention_mask, answers, soft_label in dataloader:
        image, input_id, attention_mask, answers, soft_label = \
            image.to(device), input_id.to(device), attention_mask.to(device), answers.to(device), soft_label.to(device)

        pred = model(image, input_id, attention_mask)
        loss = criterion(pred, soft_label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
        simple_acc += (pred.argmax(1) == soft_label.argmax(1)).float().mean().item()  # simple accuracy

    return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start



def eval(model, dataloader, optimizer, criterion, device):
    """
    評価用の関数．
    """
    model.eval()

    total_loss = 0
    total_acc = 0
    simple_acc = 0

    start = time.time()
    for image, input_id, attention_mask, answers, soft_label in dataloader:
        image, input_id, attention_mask, answers, soft_label = \
            image.to(device), input_id.to(device), attention_mask.to(device), answers.to(device), soft_label.to(device)

        pred = model(image, input_id, attention_mask)
        loss = criterion(pred, soft_label)

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)  # VQA accuracy
        simple_acc += (pred.argmax(1) == soft_label.argmax(1)).float().mean().item()  # simple accuracy

    return total_loss / len(dataloader), total_acc / len(dataloader), simple_acc / len(dataloader), time.time() - start

### 5. make submission file

In [ ]:
import gc

# 1. 不要になったテンソルやモデルの変数を削除（適宜変更してください）
# del model, optimizer, image, input_id, attention_mask, loss

# 2. Pythonのガベージコレクションを実行してメモリから完全に消去
gc.collect()

# 3. PyTorchが確保しているGPUのキャッシュメモリを解放
torch.cuda.empty_cache()

print("GPUメモリの解放が完了しました。")

GPUメモリの解放が完了しました。


In [ ]:
# deviceの設定
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. VQA向けの安全なData Augmentation
# 反転や大きな回転は避け、軽微なクロップと色調変化に留める
imagenet_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), # ズームインによる位置ズレ
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.0), # 軽微な色変化
    transforms.ToTensor(),
    imagenet_norm
])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_norm
])

# train_datasetにtransform_trainを適用
train_dataset = VQADataset(df_path="./data/train.json", image_dir="./data/train", transform=transform_train)
test_dataset = VQADataset(df_path="./data/valid.json", image_dir="./data/valid", transform=transform, answer=False)
test_dataset.update_dict(train_dataset)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers = 2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers = 2, pin_memory=True)

model = VQAModel(n_answer=len(train_dataset.answer2idx)).to(device)

# optimizer / criterion
num_epoch = 12
criterion = nn.BCEWithLogitsLoss()

# 3. 層ごとの学習率（Parameter Groups）の設定
# 画像・テキストの事前学習モデルは低い学習率、融合層は高めに設定する
optimizer = torch.optim.AdamW([
    {'params': model.vit.parameters(), 'lr': 5e-6},
    {'params': model.text_encoder.parameters(), 'lr': 5e-6},
    {'params': model.cross_attn.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-4}
], weight_decay=1e-4)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 234MB/s]


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# resume_model_path = "model.pt"
# if not os.path.exists(resume_model_path):
#     resume_model_path = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/model.pt"

# state_dict = torch.load(resume_model_path, map_location=device)
# model.load_state_dict(state_dict)
# print(f"Loaded checkpoint from {resume_model_path}")

num_epoch = 24

In [ ]:
from pathlib import Path
import re

checkpoint_dir = Path("/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_pattern = re.compile(r"epoch_(\d+)\.pt$")
checkpoint_files = []
for checkpoint_path in checkpoint_dir.glob("epoch_*.pt"):
    match = checkpoint_pattern.search(checkpoint_path.name)
    if match is not None:
        checkpoint_files.append((int(match.group(1)), checkpoint_path))

start_epoch = 0
if checkpoint_files:
    latest_epoch, latest_checkpoint_path = max(checkpoint_files, key=lambda item: item[0])
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        if "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = int(checkpoint.get("epoch", latest_epoch))
    else:
        model.load_state_dict(checkpoint)
        start_epoch = latest_epoch

    print(f"Loaded checkpoint from {latest_checkpoint_path} (resume from epoch {start_epoch + 1})")
else:
    print("No checkpoint found. Start training from scratch.")

for epoch in range(start_epoch, num_epoch):
    train_loss, train_acc, train_simple_acc, train_time = train(model, train_loader, optimizer, criterion, device)
    print(f"【{epoch + 1}/{num_epoch}】\n"
            f"train time: {train_time:.2f} [s]\n"
            f"train loss: {train_loss:.8f}\n"
            f"train acc: {train_acc:.4f}\n"
            f"train simple acc: {train_simple_acc:.4f}")

    checkpoint_path = checkpoint_dir / f"epoch_{epoch + 1:02d}.pt"
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_simple_acc": train_simple_acc,
        },
        checkpoint_path,
    )
    print(f"Saved checkpoint to {checkpoint_path}")


No checkpoint found. Start training from scratch.
【1/24】
train time: 493.03 [s]
train loss: 0.00052326
train acc: 0.4685
train simple acc: 0.4287
Saved checkpoint to /content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/checkpoints/epoch_01.pt
【2/24】
train time: 492.99 [s]
train loss: 0.00051427
train acc: 0.4701
train simple acc: 0.4296
Saved checkpoint to /content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/checkpoints/epoch_02.pt
【3/24】
train time: 493.35 [s]
train loss: 0.00050567
train acc: 0.4730
train simple acc: 0.4328
Saved checkpoint to /content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/checkpoints/epoch_03.pt
【4/24】
train time: 493.44 [s]
train loss: 0.00049684
train acc: 0.4725
train simple acc: 0.4321
Saved checkpoint to /content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/checkpoints/epoch_04.pt
【5/24】
train time: 493.57 [s]
train loss: 0.00048800
train acc: 0.4761
train simpl

In [25]:
# make submission file
model.eval()
submission = []
for image, input_ids, attention_mask in test_loader:
    image, input_ids, attention_mask = image.to(device), input_ids.to(device), attention_mask.to(device)
    pred = model(image, input_ids, attention_mask)
    pred = pred.argmax(1).cpu().item()
    submission.append(pred)

submission = [train_dataset.idx2answer[id] for id in submission]
submission = np.array(submission)
torch.save(model.state_dict(), "model.pt")
drive_model_path = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/model.pt"
os.makedirs(os.path.dirname(drive_model_path), exist_ok=True)
torch.save(model.state_dict(), drive_model_path)
np.save("submission.npy", submission)

## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (VQA)」から提出してください．

- `submission.npy`
- `model.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [26]:
from zipfile import ZipFile

zip_save_path = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/submission.zip"
model_path = "model.pt"
notebook_path = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/DL_Basic_2026_Spring_Competition_VQA_baseline.ipynb"

with ZipFile(zip_save_path, "w") as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path, arcname="DL_Basic_2026_Spring_Competition_VQA_baseline.ipynb")

In [27]:
import zipfile
import datetime
import os

zip_file_path = "/content/drive/MyDrive/Colab Notebooks/DLBasic2026/Final_Assignment/baseline/submission.zip"

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zf:
        print(f"{os.path.basename(zip_file_path)} の中身:")
        for info in zf.infolist():
            mod_time = datetime.datetime(*info.date_time)
            print(f"- ファイル名: {info.filename}, サイズ: {info.file_size} bytes, 更新日時: {mod_time}")
else:
    print(f"{zip_file_path} が見つかりません。")

submission.zip の中身:
- ファイル名: submission.npy, サイズ: 755416 bytes, 更新日時: 2026-06-23 20:57:30
- ファイル名: model.pt, サイズ: 1173824611 bytes, 更新日時: 2026-06-23 20:57:26
- ファイル名: DL_Basic_2026_Spring_Competition_VQA_baseline.ipynb, サイズ: 87922 bytes, 更新日時: 2026-06-23 20:53:42
